# Create a new bucket (resource version)

* [AWS resources](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/resources.html)
* [S3 resources](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html#resources)

https://boto3.amazonaws.com/v1/documentation/api/latest/guide/s3-example-creating-buckets.html

In [21]:
import sys
from dotenv import load_dotenv
import os

load_dotenv(override=True)

if "../app/" not in sys.path:
    sys.path.append("../app/")

print(sys.path)
# print(os.getenv("OPENAI_API_KEY"))

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages', '../app/']


In [20]:
import boto3
from boto3.session import Session
from dotenv import load_dotenv
import os, sys

load_dotenv(override=True)

if "../app/" not in sys.path:
    sys.path.append("../app/")

print(sys.path)
# print(os.getenv("OPENAI_API_KEY"))

bucket_name = "content-tagging-lms-lambda"

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/ryanwtsai/repos/lms-ai-agent/notebooks/venv/lib/python3.12/site-packages', '../app/']


In [ ]:
# Create a session to override your default credentials (and the default session) with Beam Data credentials

session = Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION"),
)

In [ ]:
# Create an S3 resource: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/service-resource/index.html

s3_resource = session.resource("s3")
print(type(s3_resource))

<class 'boto3.resources.factory.s3.ServiceResource'>


In [ ]:
# Create a bucket sub-resource: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucket/index.html

bucket = s3_resource.create_bucket(Bucket=bucket_name)
bucket.wait_until_exists()

In [ ]:
# Create a collection of Bucket resources: https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/service-resource/buckets.html
# What is a collection? https://boto3.amazonaws.com/v1/documentation/api/latest/guide/collections.html

buckets = s3_resource.buckets
print(type(buckets))

# Check that your new bucket exists
for b in buckets.all():
    print(b)
    # print(b.name)

<class 'boto3.resources.collection.s3.bucketsCollectionManager'>
s3.Bucket(name='aws-logs-833659032354-us-east-1')
s3.Bucket(name='content-tagging-lms')
s3.Bucket(name='content-tagging-lms-lambda')
s3.Bucket(name='do-not-delete-ssm-diagnosis-833659032354-ca-central-1-jd2k2')
s3.Bucket(name='indeed-scrape')
s3.Bucket(name='jade-stack')
s3.Bucket(name='jade-youtube')
s3.Bucket(name='lms-analytics-2')
s3.Bucket(name='lms-vimeo-transcripts')
s3.Bucket(name='project-jade-youtube')
s3.Bucket(name='sql-haystack')
s3.Bucket(name='test-bucket-d2024')
s3.Bucket(name='test-jade-transcription')


## Create a bucket notification

In [ ]:
# https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucketnotification/index.html

bucket_notification = s3_resource.BucketNotification(bucket_name)
print(type(bucket_notification))

<class 'boto3.resources.factory.s3.BucketNotification'>


In [8]:
bucket_notification.lambda_function_configurations

In [ ]:
# https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/bucketnotification/put.html

# Get bucket objects

In [12]:
bucket = s3_resource.Bucket("content-tagging-lms")
print(bucket.bucket_region)
print(bucket.creation_date)

None
2025-02-18 22:39:12+00:00


In [13]:
for o in bucket.objects.limit(10):
    print(o)

s3.ObjectSummary(bucket_name='content-tagging-lms', key='4350_vimeo_videos.parquet')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='4350_vimeo_videos_cleaned.parquet')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/AWS VPC & Networking.pptx')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Chapter_Intro.md')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Deploy Lambda Function from S3.md')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Installing DBT on Ubuntu EC2.md')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Installing H2O on Ubuntu EC2.md')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Introduction to AWS.pptx')
s3.ObjectSummary(bucket_name='content-tagging-lms', key='Content/AWS introduction/Introduction to Cloud.pptx')
s3.ObjectSummary(bucket_name='content-tag

In [15]:
o = s3_resource.Object(bucket_name="content-tagging-lms", key='Content/AWS introduction/Deploy Lambda Function from S3.md')
print(type(o))

<class 'boto3.resources.factory.s3.Object'>


In [17]:
o.content_type

'binary/octet-stream'

# Create the lambda function

In [ ]:
# Original code: https://docs.aws.amazon.com/lambda/latest/dg/with-s3-example.html

import os
import json
import urllib.parse
import boto3
from pathlib import Path
from haystack_utilities.pipelines import indexing_pipeline_lambda
from haystack_utilities.tools import MilvusContextManager

print("Loading environment variables.")

if not(os.environ.get("OPENAI_API_KEY")):
    raise Exception("Please provide an OPENAI_API_KEY environment variable.")

if not(os.environ.get("ZILLIZ_CLUSTER_ENDPOINT")):
    raise Exception("Please provide a ZILLIZ_CLUSTER_ENDPOINT environment variable (uri of your Zilliz cluster).")

if not(os.environ.get("ZILLIZ_CLUSTER_TOKEN")):
    raise Exception("Please provide a ZILLIZ_CLUSTER_TOKEN environment variable.")

if not (collection_name := os.environ.get("COLLECTION_NAME", None)):
    raise Exception("Please provide a str COLLECTION_NAME environment variable. This is the name of your Zilliz collection.")

splitting_options = os.environ.get("SPLITTING_OPTIONS", None)
if not splitting_options:
    # sentence-transformers/all-mpnet-base-v2
    # Max tokens = 384
    # Roughly 288 words (3 words = 4 tokens)
    splitting_options = {
        "split_strategy": "fixed",
        "split_by": "word",
        "split_length": 250,
        "split_overlap": 50,
        "split_threshold": 30,
        "respect_sentence_boundary": True
    }
else:
    try:
        json.loads(splitting_options)
    except json.JSONDecodeError as e:
        raise Exception("Please provide SPLITTING_OPTIONS environment variable in valid JSON format. This defines the chunking strategy.")

skip_cleaner = os.environ.get("SKIP_CLEANER", True)
add_tagger = os.environ.get("ADD_TAGGER", True)
max_content_len_chars = os.environ.get("MAX_CHUNK_LENGTH_IN_CHARS", 65535)

print("Building indexing pipeline.")
drop_old = True # If collection doesn't exist, create a new collection
with MilvusContextManager() as client:
    if collection_name in client.list_collections():
        drop_old = False # If collection exists, do not create a new collection
index_pipe = indexing_pipeline_lambda.build_indexing_pipeline(collection_name, splitting_options, skip_cleaner=skip_cleaner, add_tagger=add_tagger, max_content_len_chars=max_content_len_chars, drop_old=True)

print("Initializing AWS S3 connection.")
s3 = boto3.resource("s3")

print("Loading function.")
def lambda_handler(event, context):
    print("Received event: " + json.dumps(event, indent=2))

    # For now, the function is only expected to process one file at a time (typical S3 -> Lambda pipeline), so there is only one Record
    # Update later if batch processing is desired
    bucket = event['Records'][0]['s3']['bucket']['name']
    key = urllib.parse.unquote_plus(event['Records'][0]['s3']['object']['key'], encoding='utf-8')

    
    # Download file to /tmp
    # Initialize pipeline
    # Pass file path to pipeline
    # Environment variables
    try:
        response = s3.get_object(Bucket=bucket, Key=key)
        print("CONTENT TYPE: " + response['ContentType'])
        return response['ContentType']
    except Exception as e:
        print(e)
        print('Error getting object {} from bucket {}. Make sure they exist and your bucket is in the same region as this function.'.format(key, bucket))
        raise e

# Create a new bucket (client version)

In [17]:
region_name = os.getenv("AWS_REGION")
s3_client = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=region_name,
)

print(type(s3_client))

<class 'botocore.client.S3'>


In [20]:
# List buckets

for b in s3_client.list_buckets()["Buckets"]:
    print(b["Name"])

aws-logs-833659032354-us-east-1
content-tagging-lms
content-tagging-lms-lambda
do-not-delete-ssm-diagnosis-833659032354-ca-central-1-jd2k2
indeed-scrape
jade-stack
jade-youtube
lms-analytics-2
lms-vimeo-transcripts
project-jade-youtube
sql-haystack
test-bucket-d2024
test-jade-transcription


In [14]:
import re

bucket_name = "content-tagging-lms"
pattern = re.compile(r"\bLLM\b")
pattern = re.compile(r".pdf")
for o in s3_client.list_objects(Bucket=bucket_name)["Contents"]:
    if pattern.search(o["Key"]):
        print(o["Key"])

Content/Excel Fundamental (self-paced)/Excel.pdf
Content/Excel Fundamental (self-paced)/FinTech/Intro to FinTech.pdf
Content/Excel Fundamental (self-paced)/Introduction to Excel.pdf


In [1]:
# How to create notification?

import logging
import boto3
from botocore.exceptions import ClientError

# try:
#     # s3_client.create_bucket(Bucket="content-tagging-lms-lambda",
#     #                         CreateBucketConfiguration={"LocationConstraint": region_name})
#     s3_client.create_bucket(Bucket="content-tagging-lms-lambda")
# except ClientError as e:
#     logging.error(e)

# for b in s3_client.list_buckets()["Buckets"]:
#     print(b["Name"])

In [8]:
s3 = boto3.resource("s3")

In [ ]:
s3.BucketNotification

s3.ServiceResource()

In [24]:
type(s3)

boto3.resources.factory.s3.ServiceResource

In [25]:
dir(s3)

['Bucket',
 'BucketAcl',
 'BucketCors',
 'BucketLifecycle',
 'BucketLifecycleConfiguration',
 'BucketLogging',
 'BucketNotification',
 'BucketPolicy',
 'BucketRequestPayment',
 'BucketTagging',
 'BucketVersioning',
 'BucketWebsite',
 'MultipartUpload',
 'MultipartUploadPart',
 'Object',
 'ObjectAcl',
 'ObjectSummary',
 'ObjectVersion',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'buckets',
 'create_bucket',
 'get_available_subresources',
 'meta']

In [9]:
for b in s3.buckets.all():
    print(b)

s3.Bucket(name='ryan-demo-bucket22cd1de61e40491a88da48bcae16fdef')
s3.Bucket(name='ryan-end-to-end')


In [31]:
bucket = s3.Bucket(name='content-tagging-lms-lambda')

In [32]:
bucket.delete()

{'ResponseMetadata': {'RequestId': 'CX9MY0BANSZEAQDE',
  'HostId': 'HqBcWI/07cuXNpZkrGqfpYLrjx69Uql8gpBBDKK68XEm3aFyZzksNj3yrJpO2JHFvnuLsSRUCYo=',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'HqBcWI/07cuXNpZkrGqfpYLrjx69Uql8gpBBDKK68XEm3aFyZzksNj3yrJpO2JHFvnuLsSRUCYo=',
   'x-amz-request-id': 'CX9MY0BANSZEAQDE',
   'date': 'Fri, 04 Apr 2025 23:18:16 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 1}}

In [33]:
bucket.wait_until_not_exists()

In [35]:
bucket.bucket_region